# 06 — Analise de Regressao OLS com Variaveis Dummy e Interacoes

## Objetivo
Regressao linear com variaveis dummy: coeficientes diretamente interpretaveis.
Testamos interacoes explicitas (canal x duracao, canal x assunto) para validar o diagnostico de roteamento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
import os

RANDOM_STATE = 42
TEST_SIZE = 0.2

csv_path = 'data/customer_support_tickets.csv'
df_raw = pd.read_csv(csv_path)
print(f'Dataset: {len(df_raw)} tickets')

df = df_raw[df_raw['Ticket Status'] == 'Closed'].copy()
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'], errors='coerce')
df['duration_hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds().abs() / 3600
df = df.dropna(subset=['Customer Satisfaction Rating', 'duration_hours'])

df = df.rename(columns={
    'Ticket Channel': 'channel', 'Ticket Subject': 'subject',
    'Ticket Priority': 'priority', 'Ticket Type': 'type',
    'Customer Satisfaction Rating': 'csat', 'Customer Age': 'age',
})
df['age_bucket'] = pd.cut(df['age'], bins=[0,25,35,45,55,100], labels=['18-25','26-35','36-45','46-55','56-70'])

y = df['csat'].values
y_binned = np.round(y).astype(int)
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_binned)
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()
print(f'Treino: {len(df_train)} | Teste: {len(df_test)}')
print(f'CSAT treino: {df_train["csat"].mean():.3f} | teste: {df_test["csat"].mean():.3f}')

## Secao 2: Execucao dos 8 Experimentos

In [ ]:
EXPERIMENTS = [
    {'name': 'R1: Canal+Assunto', 'formula': 'csat ~ C(channel) + C(subject)'},
    {'name': 'R2: +Prioridade+Tipo', 'formula': 'csat ~ C(channel) + C(subject) + C(priority) + C(type)'},
    {'name': 'R3: +Duracao', 'formula': 'csat ~ C(channel) + C(subject) + C(priority) + C(type) + duration_hours'},
    {'name': 'R4: +Idade', 'formula': 'csat ~ C(channel) + C(subject) + C(priority) + C(type) + C(age_bucket)'},
    {'name': 'R5: Canal x Assunto', 'formula': 'csat ~ C(channel) * C(subject)'},
    {'name': 'R6: Duracao x Canal', 'formula': 'csat ~ C(channel) + C(subject) + C(priority) + C(type) + duration_hours:C(channel) + duration_hours'},
    {'name': 'R7: Duracao x Assunto', 'formula': 'csat ~ C(channel) + C(subject) + C(priority) + C(type) + duration_hours:C(subject) + duration_hours'},
    {'name': 'R8: Completo', 'formula': 'csat ~ C(channel) * C(subject) + duration_hours:C(channel) + C(priority) + C(type) + C(age_bucket) + duration_hours'},
]

results = []
for i, exp in enumerate(EXPERIMENTS):
    print('=' * 60)
    print(f'[{i+1}/8] {exp["name"]}')
    print(f'Formula: {exp["formula"]}')
    try:
        model = smf.ols(exp['formula'], data=df_train).fit()
        y_pred = model.predict(df_test)
        y_te = df_test['csat'].values
        mae = mean_absolute_error(y_te, y_pred)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2 = r2_score(y_te, y_pred)
        sig = model.pvalues[model.pvalues < 0.05]
        results.append({'name': exp['name'], 'mae': mae, 'rmse': rmse, 'r2_test': r2,
                        'r2_adj': model.rsquared_adj, 'aic': model.aic, 'bic': model.bic,
                        'n_params': len(model.params), 'n_significant': len(sig), 'model': model})
        print(f'MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f} R2_adj={model.rsquared_adj:.4f} Sig={len(sig)}')
    except Exception as e:
        print(f'ERRO: {e}')
        results.append({'name': exp['name'], 'mae': None, 'model': None})
print('=' * 60)
print('Concluido.')

## Secao 3: Comparacao dos Modelos

In [ ]:
valid = [r for r in results if r['mae'] is not None]
res_df = pd.DataFrame([{k:v for k,v in r.items() if k!='model'} for r in valid]).sort_values('mae')
print('Ranking por MAE:')
print(res_df[['name','mae','rmse','r2_test','r2_adj','n_params','n_significant']].to_string(index=False, float_format='%.4f'))

baseline_mae = mean_absolute_error(df_test['csat'].values, np.full(len(df_test), df_train['csat'].mean()))
print(f'Baseline (media): MAE={baseline_mae:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
names = [r['name'] for r in valid]
axes[0].barh(names, [r['mae'] for r in valid], color='#3b82f6')
axes[0].axvline(x=baseline_mae, color='red', linestyle='--', label=f'Baseline: {baseline_mae:.3f}')
axes[0].set_title('MAE por Modelo')
axes[0].legend()
axes[0].invert_yaxis()
axes[1].barh(names, [r.get('r2_adj',0) or 0 for r in valid], color='#10b981')
axes[1].set_title('R2 Ajustado')
axes[1].invert_yaxis()
axes[2].barh(names, [r.get('n_significant',0) or 0 for r in valid], color='#f97316')
axes[2].set_title('Coeficientes Significativos (p<0.05)')
axes[2].invert_yaxis()
plt.tight_layout()
plt.savefig('process-log/screenshots/p5_regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 4: Coeficientes de Canal

In [ ]:
r1 = results[0]['model']
if r1:
    params = r1.params
    pvals = r1.pvalues
    ch_coefs = {k:v for k,v in params.items() if 'channel' in k.lower()}
    ch_pvals = {k:v for k,v in pvals.items() if 'channel' in k.lower()}
    print('Coeficientes de Canal (R1):')
    for k in sorted(ch_coefs.keys()):
        sig = '***' if ch_pvals[k]<0.001 else '**' if ch_pvals[k]<0.01 else '*' if ch_pvals[k]<0.05 else ''
        print(f'  {k}: {ch_coefs[k]:+.4f} (p={ch_pvals[k]:.4f}) {sig}')

    ch_names = [k.split('[')[1].rstrip(']') if '[' in k else k for k in sorted(ch_coefs.keys())]
    ch_vals = [ch_coefs[k] for k in sorted(ch_coefs.keys())]
    plt.figure(figsize=(10, 5))
    colors_bar = ['#ef4444' if v < 0 else '#22c55e' for v in ch_vals]
    plt.barh(ch_names, ch_vals, color=colors_bar)
    plt.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    plt.xlabel('Efeito no CSAT (vs categoria base)')
    plt.title('Efeito de Cada Canal no CSAT')
    plt.tight_layout()
    plt.savefig('process-log/screenshots/p5_reg_channel_effects.png', dpi=150, bbox_inches='tight')
    plt.show()

## Secao 5: Coeficientes de Assunto

In [ ]:
if r1:
    subj_coefs = {k:v for k,v in params.items() if 'subject' in k.lower()}
    subj_pvals = {k:v for k,v in pvals.items() if 'subject' in k.lower()}
    subj_df = pd.DataFrame({'subject': list(subj_coefs.keys()), 'coef': list(subj_coefs.values())})
    subj_df['clean'] = subj_df['subject'].apply(lambda x: x.split('[')[1].rstrip(']') if '[' in x else x)
    subj_df = subj_df.sort_values('coef')
    plt.figure(figsize=(12, 8))
    colors_bar = ['#ef4444' if v < 0 else '#22c55e' for v in subj_df['coef']]
    plt.barh(subj_df['clean'], subj_df['coef'], color=colors_bar)
    plt.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    plt.xlabel('Efeito no CSAT (vs categoria base)')
    plt.title('Efeito de Cada Assunto no CSAT')
    plt.tight_layout()
    plt.savefig('process-log/screenshots/p5_reg_subject_effects.png', dpi=150, bbox_inches='tight')
    plt.show()

## Secao 6: Interacao Duracao x Canal (R6)

Pergunta crucial: 1 hora a mais muda o CSAT em quanto, em cada canal?

In [ ]:
r6 = results[5]['model'] if results[5]['mae'] else None
if r6:
    p6 = r6.params
    pv6 = r6.pvalues
    dur_coefs = {k:v for k,v in p6.items() if 'duration' in k.lower()}
    dur_pvals = {k:v for k,v in pv6.items() if 'duration' in k.lower()}
    print('Efeito de Duracao por Canal (R6):')
    for k in sorted(dur_coefs.keys()):
        sig = '***' if dur_pvals[k]<0.001 else '**' if dur_pvals[k]<0.01 else '*' if dur_pvals[k]<0.05 else ''
        print(f'  {k}: {dur_coefs[k]:+.4f} (p={dur_pvals[k]:.4f}) {sig}')
    print('Interpretacao: negativo = mais tempo piora CSAT (ACELERAR)')
    print('              positivo = mais tempo melhora CSAT (DESACELERAR)')

    dur_ch = {k:v for k,v in dur_coefs.items() if ':' in k or k == 'duration_hours'}
    if dur_ch:
        plt.figure(figsize=(10, 5))
        ch_n = [k.split('[')[1].split(']')[0] if '[' in k else 'Base' for k in sorted(dur_ch.keys())]
        ch_v = [dur_ch[k] for k in sorted(dur_ch.keys())]
        colors_bar = ['#ef4444' if v < 0 else '#3b82f6' for v in ch_v]
        plt.barh(ch_n, ch_v, color=colors_bar)
        plt.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
        plt.xlabel('Efeito de +1 hora no CSAT')
        plt.title('Efeito da Duracao no CSAT por Canal')
        plt.tight_layout()
        plt.savefig('process-log/screenshots/p5_reg_duration_by_channel.png', dpi=150, bbox_inches='tight')
        plt.show()

    # Scatter por canal
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for i, ch in enumerate(sorted(df['channel'].unique())):
        ax = axes[i//2, i%2]
        ch_data = df_train[df_train['channel'] == ch]
        ax.scatter(ch_data['duration_hours'], ch_data['csat'], alpha=0.3, s=20, color='#6b7280')
        z = np.polyfit(ch_data['duration_hours'], ch_data['csat'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(ch_data['duration_hours'].min(), ch_data['duration_hours'].max(), 100)
        ax.plot(x_line, p(x_line), color='#ef4444', linewidth=2, label=f'slope={z[0]:.4f}')
        r, pval = pearsonr(ch_data['duration_hours'], ch_data['csat'])
        ax.set_title(f'{ch} (r={r:.3f}, p={pval:.3f})')
        ax.set_xlabel('Duracao (horas)')
        ax.set_ylabel('CSAT')
        ax.legend()
    plt.suptitle('Duracao vs CSAT por Canal', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('process-log/screenshots/p5_reg_duration_scatter_by_channel.png', dpi=150, bbox_inches='tight')
    plt.show()

## Secao 7: Interacao Duracao x Assunto (R7)

In [ ]:
r7 = results[6]['model'] if results[6]['mae'] else None
if r7:
    p7 = r7.params
    pv7 = r7.pvalues
    dur_subj = {k:v for k,v in p7.items() if 'duration' in k.lower() and ':' in k}
    dur_subj_p = {k:v for k,v in pv7.items() if 'duration' in k.lower() and ':' in k}
    base_dur = p7.get('duration_hours', 0)
    print(f'Efeito base duracao: {base_dur:.4f}')
    subj_eff = []
    for k in sorted(dur_subj.keys()):
        total = base_dur + dur_subj[k]
        name = k.split('[')[1].split(']')[0] if '[' in k else k
        sig = '*' if dur_subj_p[k] < 0.05 else ''
        print(f'  {name}: interacao={dur_subj[k]:+.4f} total={total:+.4f} p={dur_subj_p[k]:.4f} {sig}')
        subj_eff.append({'subject': name, 'total': total, 'p': dur_subj_p[k]})

    eff_df = pd.DataFrame(subj_eff).sort_values('total')
    plt.figure(figsize=(14, 8))
    colors_bar = ['#ef4444' if v < 0 else '#3b82f6' for v in eff_df['total']]
    plt.barh(eff_df['subject'], eff_df['total'], color=colors_bar)
    for i, (_, row) in enumerate(eff_df.iterrows()):
        if row['p'] < 0.05:
            plt.text(row['total'], i, ' *', va='center', fontweight='bold', fontsize=12)
    plt.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    plt.xlabel('Efeito de +1 hora no CSAT (total)')
    plt.title('Efeito da Duracao por Assunto (* = significativo)')
    plt.tight_layout()
    plt.savefig('process-log/screenshots/p5_reg_duration_by_subject.png', dpi=150, bbox_inches='tight')
    plt.show()

## Secao 8: Validacao com Diagnostico Manual

In [ ]:
REDIRECT_VIABLE = {
    'Email': {'Phone': True, 'Chat': True, 'Social media': False},
    'Chat': {'Phone': True, 'Email': True, 'Social media': False},
    'Phone': {'Email': True, 'Chat': True, 'Social media': False},
    'Social media': {'Email': True, 'Phone': True, 'Chat': False},
}
channels = sorted(df['channel'].unique())
subjects = sorted(df['subject'].unique())
subj_ch_csat = {}
for sub in subjects:
    ch_stats = []
    for ch in channels:
        pd2 = df[(df['channel']==ch)&(df['subject']==sub)]
        if len(pd2)>=3: ch_stats.append({'channel':ch,'avg_csat':round(pd2['csat'].mean(),2)})
    subj_ch_csat[sub] = sorted(ch_stats, key=lambda x:-x['avg_csat'])

diags = []
for ch in channels:
    for sub in subjects:
        pd2 = df[(df['channel']==ch)&(df['subject']==sub)]
        if len(pd2)<3: continue
        rp,_ = pearsonr(pd2['duration_hours'], pd2['csat'])
        rp = round(rp,3)
        pc = round(pd2['csat'].mean(),2)
        chs = subj_ch_csat.get(sub,[])
        bc = chs[0] if chs else None
        if rp<-0.3: sc='acelerar'
        elif rp>0.3: sc='desacelerar'
        else:
            gap = bc['avg_csat']-pc if bc else 0
            if gap>0.5 and bc and bc['channel']!=ch:
                vi=[c for c in chs if c['channel']!=ch and c['avg_csat']>pc+0.3 and REDIRECT_VIABLE.get(ch,{}).get(c['channel'],False)]
                sc='redirecionar' if vi else 'quarentena'
            elif pc>=3.5: sc='manter'
            else:
                ab=all(c['avg_csat']<3.0 for c in chs)
                sc='quarentena' if(ab and pc<3.0) else 'liberar'
        diags.append({'channel':ch,'subject':sub,'scenario':sc,'r_pair':rp,'pair_csat':pc})

diag_df = pd.DataFrame(diags)
print(f'Diagnostico: {len(diag_df)} pares')
print(diag_df['scenario'].value_counts().to_string())

sc_colors = {'acelerar':'#ef4444','desacelerar':'#3b82f6','redirecionar':'#f97316',
             'quarentena':'#eab308','manter':'#22c55e','liberar':'#9ca3af'}
plt.figure(figsize=(12, 8))
for sc, col in sc_colors.items():
    sd = diag_df[diag_df['scenario']==sc]
    plt.scatter(sd['r_pair'], sd['pair_csat'], c=col, label=sc, s=80, alpha=0.7, edgecolors='black', linewidths=0.5)
plt.axvline(x=-0.3, color='red', linestyle=':', alpha=0.5)
plt.axvline(x=0.3, color='blue', linestyle=':', alpha=0.5)
plt.axhline(y=3.5, color='green', linestyle=':', alpha=0.5)
plt.xlabel('Pearson r (duracao x CSAT)')
plt.ylabel('CSAT Medio')
plt.title('Mapa dos 64 Pares: r vs CSAT por Cenario')
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig('process-log/screenshots/p5_reg_diagnostic_map.png', dpi=150, bbox_inches='tight')
plt.show()

## Secao 9: Conclusoes

In [ ]:
print('=' * 60)
print('CONCLUSOES — REGRESSAO OLS')
print('=' * 60)
best = min(valid, key=lambda r: r['mae'])
print(f'1. MELHOR MODELO: {best["name"]}')
print(f'   MAE={best["mae"]:.4f}, R2_adj={best.get("r2_adj",0):.4f}')
print(f'   Params={best.get("n_params",0)}, Significativos={best.get("n_significant",0)}')
print(f'2. BASELINE MAE: {baseline_mae:.4f}')
print(f'   Melhoria: {((baseline_mae-best["mae"])/baseline_mae*100):.1f}%')
print('3. INTERPRETACAO:')
print('   Os coeficientes de regressao fornecem evidencia direta')
print('   para decisoes de roteamento por canal e priorizacao de fila.')
print('   Comparar com resultados do GBR+SHAP (notebook 05).')
print('=' * 60)